# Unicorn Data Explorer

Interactive exploration of NBA play-by-play data while the transformer trains.

**Sections:**
1. The 3-Point Revolution (temporal context for embeddings)
2. Player Volume & Coverage (who has good/bad embeddings)
3. Outcome Distributions by Game State (validating model inputs)
4. CBOW Embedding Analysis (the "before" picture)
5. Training Split Analysis (understanding distribution shift)

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_context("talk")
sns.set_style("whitegrid")

PARQUET = "possessions.parquet"
LOOKUP = "player_season_lookup.csv"
CBOW_CKPT = "cbow_checkpoint.pt"

In [ ]:
# Load data (select columns to save memory)
cols = [
    "game_id", "season", "outcome",
    "sec_remaining_game", "score_diff", "period",
    "home_score", "away_score", "homename", "awayname",
] + [f"p{i}" for i in range(10)]

df = pd.read_parquet(PARQUET, columns=cols)
df_clean = df[df["outcome"] != "other"].copy()
print(f"Raw: {len(df):,} | Filtered: {len(df_clean):,} possessions")

lu = pd.read_csv(LOOKUP)
id_to_name = lu.set_index("player_season_id")["player"].to_dict()
id_to_label = {row.player_season_id: f"{row.player} ({row.season})"
               for _, row in lu.iterrows()}
print(f"{len(lu):,} player-season IDs, {lu['player'].nunique():,} unique players")

---
## 1. The 3-Point Revolution

The NBA shifted dramatically toward 3-point shooting after ~2015. This matters for our model because the test set (2021+) comes from a fundamentally different basketball era than most of the training data (<2019).

In [ ]:
# Outcome share by season
share = (
    df_clean.groupby(["season", "outcome"]).size()
    .groupby(level=0).apply(lambda s: s / s.sum())
    .unstack(fill_value=0)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
share.plot.area(ax=ax, colormap="tab10", alpha=0.85)
ax.axvline(2015, color="red", ls="--", alpha=0.5, label="Curry MVP season")
ax.axvline(2019, color="black", ls="--", alpha=0.5, label="Train/val split")
ax.set_title("How NBA Basketball Changed: Outcome Shares by Season")
ax.set_ylabel("Fraction of possessions")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 3pt attempt rate and efficiency by season
season_stats = df_clean.groupby("season")["outcome"].value_counts().unstack(fill_value=0)
season_totals = season_stats.sum(axis=1)

three_attempts = season_stats.get("made_3pt", 0) + season_stats.get("missed_3pt", 0)
three_rate = three_attempts / season_totals
three_eff = season_stats.get("made_3pt", 0) / three_attempts.clip(lower=1)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(three_rate.index, three_rate.values * 100, "b-o", markersize=4, label="3pt attempt rate")
ax1.set_ylabel("3pt attempt rate (%)", color="b")
ax1.set_xlabel("Season")

ax2 = ax1.twinx()
ax2.plot(three_eff.index, three_eff.values * 100, "r-s", markersize=4, label="3pt make rate")
ax2.set_ylabel("3pt make rate (%)", color="r")

ax1.axvline(2015, color="gray", ls="--", alpha=0.3)
ax1.set_title("3-Point Revolution: More Attempts AND Better Efficiency")
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

In [ ]:
# Pace proxy: possessions per game by season
pace = df.groupby(["season", "game_id"]).size().groupby("season").mean()

fig, ax = plt.subplots(figsize=(12, 4))
pace.plot(ax=ax, marker="o", markersize=4)
ax.set_title("NBA Pace: Possessions per Game by Season")
ax.set_ylabel("Possessions / game")
ax.set_xlabel("Season")
ax.axvline(2019, color="black", ls="--", alpha=0.3, label="Train/val boundary")
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Player Volume & Coverage

Embedding quality is proportional to data volume. Starters with 10K+ possessions per season will have strong embeddings; 10-day contract players with <100 possessions will be noise.

In [ ]:
# Possession volume per player-season ID
ply_cols = [f"p{i}" for i in range(10)]
vol = df_clean[ply_cols].stack().value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram (log scale)
axes[0].hist(vol.values, bins=100, edgecolor="none", alpha=0.7)
axes[0].set_xscale("log")
axes[0].axvline(vol.median(), color="red", ls="--", label=f"Median: {vol.median():,.0f}")
axes[0].axvline(100, color="orange", ls="--", label=f"<100: {(vol < 100).sum():,} IDs")
axes[0].set_title("Player-Season Possession Volume")
axes[0].set_xlabel("Possessions (log scale)")
axes[0].set_ylabel("Count of player-season IDs")
axes[0].legend()

# Top 25
top25 = vol.head(25)
top25.index = [id_to_label.get(i, str(i)) for i in top25.index]
top25.sort_values().plot.barh(ax=axes[1])
axes[1].set_title("Top 25 Player-Seasons by Volume")
axes[1].set_xlabel("Possessions")

plt.tight_layout()
plt.show()

print(f"Total player-season IDs: {len(vol):,}")
print(f"IDs with <100 possessions: {(vol < 100).sum():,} ({(vol < 100).mean():.1%})")
print(f"IDs with <500 possessions: {(vol < 500).sum():,} ({(vol < 500).mean():.1%})")

In [ ]:
# Career span distribution
career_spans = lu.groupby("player")["season"].agg(["count", "min", "max"])
career_spans.columns = ["seasons", "first", "last"]

fig, ax = plt.subplots(figsize=(10, 4))
career_spans["seasons"].hist(bins=range(1, 25), ax=ax, edgecolor="white", alpha=0.7)
ax.set_title("Career Span Distribution (seasons in dataset)")
ax.set_xlabel("Number of seasons")
ax.set_ylabel("Number of players")

# Annotate longest careers
longest = career_spans.nlargest(5, "seasons")
for _, row in longest.iterrows():
    print(f"  {row.name}: {row.seasons} seasons ({row.first}-{row.last})")

# Prior-year init coverage
multi = career_spans[career_spans["seasons"] >= 2]
print(f"\nPlayers with 2+ seasons (benefit from prior-year init): "
      f"{len(multi):,} / {len(career_spans):,} ({len(multi)/len(career_spans):.1%})")

plt.tight_layout()
plt.show()

---
## 3. Outcome Distributions by Game State

Do possession outcomes vary with game state (score_diff, time remaining, period)? If yes, game state is a meaningful model input. If not, it's noise.

In [ ]:
# Outcome distribution by score differential
bins = [-50, -15, -10, -5, -2, 2, 5, 10, 15, 50]
labels = ["-15+", "-10:-15", "-5:-10", "-2:-5", "close", "+2:+5", "+5:+10", "+10:+15", "+15+"]
df_clean["score_bin"] = pd.cut(df_clean["score_diff"], bins=bins, labels=labels)

heat_data = (
    df_clean.groupby(["score_bin", "outcome"]).size()
    .groupby(level=0).apply(lambda s: s / s.sum())
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heat_data, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax,
            cbar_kws={"label": "Fraction"})
ax.set_title("Outcome Distribution by Score Differential")
ax.set_ylabel("Score diff (home - away)")
ax.set_xlabel("Outcome")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Outcome distribution over game clock (2-minute windows)
df_clean["min_remaining"] = df_clean["sec_remaining_game"] / 60
df_clean["time_bin"] = pd.cut(df_clean["min_remaining"], bins=range(0, 52, 2))

time_dist = (
    df_clean.groupby(["time_bin", "outcome"]).size()
    .groupby(level=0).apply(lambda s: s / s.sum())
    .unstack(fill_value=0)
)
time_dist.index = [f"{int(iv.left)}-{int(iv.right)}" for iv in time_dist.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Shot types
shot_cols = ["made_3pt", "missed_3pt", "made_2pt_close", "made_2pt_mid",
             "missed_2pt_close", "missed_2pt_mid"]
time_dist[shot_cols].plot(ax=axes[0], marker=".", markersize=3)
axes[0].set_title("Shot types over game clock")
axes[0].set_xlabel("Minutes remaining")
axes[0].set_ylabel("Fraction")
axes[0].legend(fontsize=7)
axes[0].invert_xaxis()

# Non-shot outcomes
other_cols = ["FT", "turnover_live", "turnover_dead"]
time_dist[other_cols].plot(ax=axes[1], marker=".", markersize=3)
axes[1].set_title("FT & Turnovers over game clock")
axes[1].set_xlabel("Minutes remaining")
axes[1].set_ylabel("Fraction")
axes[1].legend(fontsize=8)
axes[1].invert_xaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Clutch vs non-clutch comparison
clutch_mask = (
    (df_clean["period"] >= 4) &
    (df_clean["sec_remaining_game"] < 300) &
    (df_clean["score_diff"].abs() <= 5)
)

clutch_dist = df_clean[clutch_mask]["outcome"].value_counts(normalize=True)
normal_dist = df_clean[~clutch_mask]["outcome"].value_counts(normalize=True)

compare = pd.DataFrame({"Clutch": clutch_dist, "Non-clutch": normal_dist})
compare = compare.reindex(["made_3pt", "missed_3pt", "made_2pt_close", "made_2pt_mid",
                           "missed_2pt_close", "missed_2pt_mid", "FT",
                           "turnover_live", "turnover_dead"])

fig, ax = plt.subplots(figsize=(12, 5))
compare.plot.bar(ax=ax, width=0.7)
ax.set_title(f"Clutch vs Non-Clutch Outcome Distribution\n"
             f"(Clutch = Q4+, <5min, score within 5 | {clutch_mask.sum():,} possessions)")
ax.set_ylabel("Fraction")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

---
## 4. CBOW Embedding Analysis — The "Before" Picture

The CBOW model mean-pools player embeddings (no player-player interaction). It achieved **12% val accuracy** on 9-class prediction (random baseline: ~11%). Let's see what its embeddings look like — this is what the transformer needs to beat.

In [ ]:
# Load CBOW embeddings
ckpt = torch.load(CBOW_CKPT, map_location="cpu", weights_only=True)
emb = ckpt["state_dict"]["emb.weight"]
emb_norm = F.normalize(emb, dim=1)
print(f"CBOW embeddings: {emb.shape}")
print(f"Best val acc: {ckpt.get('best_val_acc', 'N/A')}")
print(f"Epochs: {ckpt.get('epochs', 'N/A')}")

# Basic stats
norms = emb.norm(dim=1)
print(f"\nNorm: mean={norms.mean():.3f}, std={norms.std():.3f}")

# Average pairwise similarity (sample)
n_sample = 1000
sample = emb_norm[np.random.choice(emb.shape[0], n_sample, replace=False)]
sim_mat = sample @ sample.T
sim_mat.fill_diagonal_(0)
avg_sim = sim_mat.sum() / (n_sample * (n_sample - 1))
print(f"Avg pairwise cosine similarity (n={n_sample}): {avg_sim:.4f}")

In [ ]:
# Same-player cross-season similarity
# Does the CBOW model know that LeBron_2022 ≈ LeBron_2023?
multi_season = lu.groupby("player").filter(lambda x: len(x) >= 3)
same_player_sims = []
consecutive_sims = []

for player, group in multi_season.groupby("player"):
    pids = group.sort_values("season")["player_season_id"].values
    pids = pids[pids < emb.shape[0]]
    if len(pids) < 2:
        continue
    for i in range(len(pids)):
        for j in range(i + 1, len(pids)):
            same_player_sims.append((emb_norm[pids[i]] @ emb_norm[pids[j]]).item())
    for i in range(len(pids) - 1):
        consecutive_sims.append((emb_norm[pids[i]] @ emb_norm[pids[i+1]]).item())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(same_player_sims, bins=60, alpha=0.6, label=f"Same player, any seasons (n={len(same_player_sims):,})")
ax.hist(consecutive_sims, bins=60, alpha=0.6, label=f"Same player, consecutive (n={len(consecutive_sims):,})")
ax.axvline(avg_sim.item(), color="red", ls="--", label=f"Random pair avg: {avg_sim:.3f}")
ax.set_title("CBOW: Same-Player Cross-Season Similarity")
ax.set_xlabel("Cosine similarity")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Same-player (any): mean={np.mean(same_player_sims):.4f}")
print(f"Same-player (consecutive): mean={np.mean(consecutive_sims):.4f}")
print(f"Random baseline: {avg_sim:.4f}")
print(f"\nVerdict: {'CBOW shows temporal coherence' if np.mean(consecutive_sims) > avg_sim.item() + 0.05 else 'CBOW has NO temporal coherence — transformer must fix this'}")

In [ ]:
# Nearest neighbors for notable players
notable = {
    "LeBron James": "jamesle01",
    "Stephen Curry": "curryst01",
    "Kevin Durant": "duranke01",
    "James Harden": "hardeja01",
    "Giannis": "anMDeto01",
    "Luka Doncic": "doncilu01",
    "Nikola Jokic": "jokicni01",
}

for display_name, bbref_id in notable.items():
    rows = lu[lu["player"] == bbref_id].sort_values("season")
    if len(rows) == 0:
        continue
    latest = rows.iloc[-1]
    pid = latest.player_season_id
    if pid >= emb.shape[0]:
        continue

    sims = emb_norm @ emb_norm[pid]
    topk = sims.topk(6)
    print(f"\n{display_name} ({latest.season}):")
    for idx, sim in zip(topk.indices.tolist(), topk.values.tolist()):
        if idx == pid:
            continue
        name = id_to_label.get(idx, f"ID_{idx}")
        print(f"  {name:35s} sim={sim:.4f}")

In [ ]:
# t-SNE of CBOW embeddings (top 2000 by volume)
from sklearn.manifold import TSNE

ply_cols = [f"p{i}" for i in range(10)]
vol_all = df_clean[ply_cols].stack().value_counts()
top_ids = vol_all.head(2000).index.tolist()

emb_subset = emb[top_ids].numpy()
season_map = lu.set_index("player_season_id")["season"].to_dict()
seasons = [season_map.get(i, 2010) for i in top_ids]

print("Running t-SNE on 2000 CBOW embeddings...")
coords = TSNE(n_components=2, perplexity=40, learning_rate=200,
              init="pca", random_state=42, n_iter=1000).fit_transform(emb_subset)

fig, ax = plt.subplots(figsize=(12, 10))
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=seasons, cmap="viridis",
                     s=8, alpha=0.5)
plt.colorbar(scatter, label="Season")
ax.set_title("CBOW Embedding t-SNE (top 2000 players by volume)\n"
             "The 'before' picture — expect weak structure")

# Annotate notable players
for display_name, bbref_id in notable.items():
    rows = lu[lu["player"] == bbref_id].sort_values("season")
    if len(rows) == 0:
        continue
    latest = rows.iloc[-1]
    if latest.player_season_id in top_ids:
        idx = top_ids.index(latest.player_season_id)
        ax.annotate(display_name, (coords[idx, 0], coords[idx, 1]),
                   fontsize=8, fontweight="bold",
                   arrowprops=dict(arrowstyle="->", color="red", lw=0.5))

ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()

### CBOW Embedding Verdict

- **Val accuracy:** 12% (barely above 11% random baseline)
- **Cross-season coherence:** Near zero — the model treats each player-season as independent
- **Nearest neighbors:** Semantically meaningless — LeBron's neighbors are random bench players
- **t-SNE:** Weak structure — no clear position or role clusters

**Conclusion:** Mean-pooling destroys player interaction information. This is the baseline the attention-pooling transformer must beat. When Phase A training finishes, we'll re-run these analyses and show the improvement.

---
## 5. Training Split Analysis

Understanding what the model sees (train <2019) vs. what it's tested on (test >=2021).

In [ ]:
# Split statistics
splits = {
    "train (<2019)": df_clean[df_clean["season"] < 2019],
    "val (2019-2020)": df_clean[df_clean["season"].between(2019, 2020)],
    "test (>=2021)": df_clean[df_clean["season"] >= 2021],
}

ply_cols = [f"p{i}" for i in range(10)]
stats = []
for name, split in splits.items():
    unique_ids = set()
    for c in ply_cols:
        unique_ids.update(split[c].unique())
    stats.append({
        "Split": name,
        "Possessions": f"{len(split):,}",
        "Games": f"{split['game_id'].nunique():,}",
        "Player-season IDs": f"{len(unique_ids):,}",
        "Seasons": f"{split['season'].min()}-{split['season'].max()}",
    })

display(pd.DataFrame(stats).set_index("Split"))

In [ ]:
# Outcome distribution by split
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

class_order = ["made_3pt", "missed_3pt", "made_2pt_close", "made_2pt_mid",
               "missed_2pt_close", "missed_2pt_mid", "FT", "turnover_live", "turnover_dead"]

for ax, (name, split) in zip(axes, splits.items()):
    dist = split["outcome"].value_counts(normalize=True).reindex(class_order)
    dist.plot.bar(ax=ax, color="steelblue", alpha=0.7)
    ax.set_title(name)
    ax.set_ylabel("Fraction" if ax == axes[0] else "")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=8)

plt.suptitle("Outcome Distribution Shift Across Splits", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Player overlap: how many test-era PLAYERS were in training?
train_players = set(lu[lu["season"] < 2019]["player"])
val_players = set(lu[lu["season"].between(2019, 2020)]["player"])
test_players = set(lu[lu["season"] >= 2021]["player"])

val_overlap = val_players & train_players
test_overlap = test_players & train_players
test_new = test_players - train_players  # rookies / first appearances

print("Player (not player-season ID) overlap with training:")
print(f"  Val players in train:  {len(val_overlap):,} / {len(val_players):,} ({len(val_overlap)/len(val_players):.1%})")
print(f"  Test players in train: {len(test_overlap):,} / {len(test_players):,} ({len(test_overlap)/len(test_players):.1%})")
print(f"  Test-only (rookies/new): {len(test_new):,} ({len(test_new)/len(test_players):.1%})")
print(f"\nNote: Player-season IDs have ZERO overlap by design (each season gets a unique ID).")
print(f"Prior-year init bridges this gap: it initializes LeBron_2019 from LeBron_2018.")

---
## Next: After Transformer Training

When Phase A finishes, re-run sections 4 with transformer embeddings:
```python
python analyze_embeddings.py --ckpt pretrain_checkpoint.pt
```

Key questions to answer:
1. Do transformer nearest neighbors make basketball sense? (roles, positions)
2. Does same-player cross-season similarity improve? (temporal coherence)
3. Do position/role clusters emerge in t-SNE?
4. What do the attention weights look like? (who does the model focus on?)